# LC 213 — House Robber II
**Difficulty:** Medium | **Pattern:** 1D DP on Circular Array

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> The first and last houses are
neighbours, so you can never rob both. Break the circle by
solving the linear House Robber problem <em>twice</em>: once
excluding the last house, once excluding the first. The answer
is the max of the two results.
</div>

## Official Problem Statement

You are a professional robber planning to rob houses along a
street. All houses are **arranged in a circle**, meaning the
first house is the neighbour of the last one.

Given an integer array `nums` representing money in each house,
return the **maximum amount** you can rob tonight without
alerting the police (no two adjacent houses).

**Constraints:**
- `1 <= nums.length <= 100`
- `0 <= nums[i] <= 1000`

## What This Is Actually Asking

It is House Robber I but the street wraps around, making house 0
and house n-1 adjacent. You cannot rob both. The elegant fix:
if house 0 is robbed, house n-1 is off-limits, so solve on
`nums[0..n-2]`. If house n-1 is robbed, house 0 is off-limits,
so solve on `nums[1..n-1]`. Taking the max of both answers
covers all valid cases without any graph machinery.

## Walk Through an Example by Hand

`nums = [2, 3, 2]`  (circle: 2–3–2–2...)

```
Run 1: nums[0..1] = [2, 3]  → rob1([2,3]) = 3
Run 2: nums[1..2] = [3, 2]  → rob1([3,2]) = 3
Answer = max(3, 3) = 3
```

`nums = [1, 2, 3, 1]`
```
Run 1: [1,2,3]  → dp: 1,2,4  → 4
Run 2: [2,3,1]  → dp: 2,3,3  → 3
Answer = max(4, 3) = 4
```

## The Picture

```
nums = [2, 3, 2, 5, 4]
       idx 0  1  2  3  4

Circle constraint: house 0 and house 4 are neighbours.

Break circle into two linear problems:

  Scenario A (exclude last):  [2, 3, 2, 5]
    dp: 2, 3, 4, 8  → best = 8

  Scenario B (exclude first): [3, 2, 5, 4]
    dp: 3, 3, 8, 8  → best = 8

  Answer = max(8, 8) = 8
  (rob houses 0 and 3: 2+5=7 ... or 1 and 3: 3+5=8)

Helper = linear rob from LC 198:
  prev2, prev1 = 0, 0
  for x in subarray:
      cur = max(prev1, prev2 + x)
      prev2, prev1 = prev1, cur
  return prev1
```

## When To Use This Pattern

- When a linear DP problem is made **circular** (first and last
  element are neighbours), think **split into two linear runs**.
- When you recognise that only two boundary elements conflict,
  think **enumerate the two cases and take the max**.
- When a helper function already solves the linear version,
  think **reuse it on two subarrays**.
- When n=1 or n=2 appear as edge cases, think **handle them
  before the general logic**.
- When the constraint is "no two adjacent in a ring", this
  two-run trick almost always applies.

## The Approach

Write a helper that solves the linear (non-circular) House
Robber on any subarray using the rolling two-variable DP. Handle
edge cases: if there is only one house, return its value. Then
call the helper twice — once on `nums[:-1]` and once on
`nums[1:]` — and return the maximum.

In [1]:
from typing import List

In [2]:
def test_harness(func):
    cases = [
        # (nums, expected)
        ([2, 3, 2],              3),
        ([1, 2, 3, 1],           4),
        ([1, 2, 3],              3),
        ([0],                    0),   # single house
        ([5],                    5),   # single house
        ([1, 1],                 1),   # two houses
        ([200, 3, 140, 20, 10],  340), # rob 0 and 2
    ]
    passed = 0
    for nums, expected in cases:
        result = func(nums)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status}: nums={nums} "
                f"=> got {result}, want {expected}"
            )
    print(f"\nSummary: {passed}/{len(cases)} passed")

In [10]:
def rob(houses: List[int]) -> int:
    """
    Return max money from a circular street with no adjacency.

    Strategy:
      1. Edge case: single house → return nums[0]
      2. Define helper rob_linear(sub) using rolling DP
      3. Return max(rob_linear(nums[:-1]),
                    rob_linear(nums[1:]))

    Args:
        nums: list of non-negative integers, len >= 1
    Returns:
        Maximum loot with no two adjacent houses.
    """
    if len(houses) == 1: return houses[0]
    def rob2(nums : List[int]) -> int:
        if len(nums) == 1 : return nums[0]
        if len(nums) == 2 : return max(nums[0], nums[1])
        prev = max(nums[0], nums[1])
        pprev = nums[0]
        curr = 0
        for i in range(2, len(nums)):
            curr = max(nums[i]+ pprev, prev)
            pprev = prev
            prev = curr
        return curr  
    return max( rob2(houses[1:]), rob2(houses[:-1]) )

test_harness(rob)



Summary: 7/7 passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(rob)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute-force subsets | O(2^n) | O(n) | Infeasible |
| Single DP ignoring circle | O(n) | O(1) | Wrong answer |
| **Two linear DP runs** | **O(n)** | **O(1)** | Correct & optimal |
| DP array version | O(n) | O(n) | Correct but extra space |

## Real World Connection

Circular constraints appear constantly in finance: at **Citi**,
risk limits for a trading book wrap around so that the first and
last positions in a ring trade cannot both be at maximum size.
In **AWS** networking, ring topologies (e.g., SONET rings) have
the same head-to-tail adjacency issue when allocating bandwidth.
For a **data engineer**, partitioned Kafka topics arranged in a
ring for consumer-group assignment face the same "first and last
are neighbours" constraint. The two-run trick is a clean,
interview-ready solution to any such circular DP.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra